This notebook generates the raw plots used in Figure 2b

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import numpy as np

plt.rcParams["font.family"] = "Arial"
sns.set_theme(style="darkgrid")



Load SCD size distribution per region

In [ ]:
with open('../../../data/scd_sizes.pkl', 'rb') as f:
    scd_sizes = pickle.load(f)

size_df = pd.DataFrame(
    [{"Region": region, "SCD Size": size}
        for region, sizes in scd_sizes.items()
        for size in sizes])

Load CV results

In [ ]:
with open('../../../datastore/results/CVPerformance.pkl', 'rb') as f:
    df= pickle.load(f)

make a column to align plotting region names with names used in modelling 

In [ ]:
df["RegionOfficial"] = df["Region"].str.split(", ").str[-1]
df["RegionOfficial"] = df["RegionOfficial"].replace({"South Africa": "Karoo"})

# define region order
region_order = ['Moora, Australia', 'Stiles, TX, USA', 'Borisoglebsk, Russia', 'Coubert, France', 'Sao Fransisco Basin, Brazil', 'Karoo Basin, South Africa']

Calculate scd size distribution per region

In [ ]:
# get boxplot data in exactly the same order as bars
box_data = []

for region in df["RegionOfficial"]:
    x = np.asarray(scd_sizes[region], dtype=float)
    to_plot_region = df.loc[df["RegionOfficial"] == region, "Region"].values[0]

    # non-positive values cannot be shown on log scale
    x = x[np.isfinite(x) & (x > 0)]
    x = pd.Series(x, name=to_plot_region)
    box_data.append(x)

box_df = pd.concat(box_data, axis=1)

In [ ]:
sns.set_theme(style="darkgrid")

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(3.5, 5),
    sharex=True,
    gridspec_kw={
        "height_ratios": [2, 1],
        "hspace": 0.05
    }
)

# bar plot
sns.barplot(
    data=df,
    x="Region",
    y="F1 Score",
    order=region_order,
    ax=ax1,
    width=0.8,
    color='green'
)

ax1.set_xlabel("")
ax1.set_ylabel("F1 Score")
ax1.bar_label(ax1.containers[0], fontsize=12, fmt="%.2f", padding=3, color="black")
ax1.set_ylim(0, 1)
ax1.tick_params(axis="y", labelsize=12)

# box plot
sns.boxplot(
    data=box_df,
    order=region_order,
    ax=ax2,
    width=0.7,
    color='purple'
)

ax2.set_xlabel("")
ax2.set_ylabel("SCD radius (m)", fontsize=12)
ax2.set_ylim(8, 2000)
ax2.set_yscale('log')

for ax in [ax1, ax2]:
    for spine in ["bottom", "left", "top", "right"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color("black")

# only define the region labels once
ax2.tick_params(axis="x", which='both', rotation=45, labelsize=12, bottom=True)

tick_labels = ['Moora, Australia', 'Stiles, USA', 'Povorino, Russia', 'Coubert, France', 'SF Basin, Brazil', 'Welkom, South Africa']
ax1.set_xticklabels(tick_labels)

plt.savefig('out/Fig2b.svg')